In [1]:


# Algorithm 4 (MMJ distance by Calculation and Copy) can also be implemented 
# by sorting the edges of the MST in ascending order (from small to large). 
# This file shows how it can be accomplished.



In [2]:
import time
import pickle
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import pairwise_distances
import networkx as nx
import sys
from joblib import Parallel, delayed

In [3]:
test_data_145 = pickle.load(  open( "./data/test_data_145.p", "rb" ) ) 


In [4]:
import numpy as np
import networkx as nx
from numba import njit
from sklearn.metrics import pairwise_distances
from joblib import Parallel, delayed

# Global matrix
mmj_matrix = None  

# --- Numba-accelerated Prim's MST ---
@njit
def primMST_numba(distance_matrix):
    V = distance_matrix.shape[0]
    key = np.full(V, np.inf)
    parent = -np.ones(V, dtype=np.int64)
    inMST = np.zeros(V, dtype=np.bool_)
    key[0] = 0.0
    
    for _ in range(V):
        # Select the minimum key vertex from the set of vertices not yet included in MST
        u = -1
        min_val = np.inf
        for v in range(V):
            if (not inMST[v]) and (key[v] < min_val):
                min_val = key[v]
                u = v
        if u == -1:
            break
        inMST[u] = True
        for v in range(V):
            # Update key only if there is an edge and v is not yet in MST
            if (distance_matrix[u, v] > 0) and (not inMST[v]) and (distance_matrix[u, v] < key[v]):
                key[v] = distance_matrix[u, v]
                parent[v] = u
    return parent

def construct_MST_from_graph(distance_matrix):
    V = distance_matrix.shape[0]
    # Use the numba-accelerated Prim's algorithm to get the parent array
    parent = primMST_numba(distance_matrix)
    
    MST = nx.Graph()
    for i in range(V):
        MST.add_node(i)
    # Build the MST edges from the parent array (skip the root at index 0)
    print("Print the MST:")
    for i in range(1, V):
        MST.add_edge(parent[i], i, weight=distance_matrix[i, parent[i]])
        print(parent[i], i, distance_matrix[i, parent[i]])
    return MST

 

        
def cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(X, round_n=1, n_jobs=-1):
    global mmj_matrix  
    lenX = len(X)
    distance_matrix = np.round(pairwise_distances(X), round_n)
    mmj_matrix = np.zeros((lenX, lenX))  

    # Build the MST using the numba-accelerated routine
    MST = construct_MST_from_graph(distance_matrix)
    MST_edge_list = list(MST.edges(data='weight'))

    edge_node_list = [(edge[0], edge[1]) for edge in MST_edge_list]
    edge_weight_list = [edge[2] for edge in MST_edge_list]
    edge_small_to_large_arg = np.argsort(edge_weight_list)
    edge_weight_small_to_large = np.sort(edge_weight_list)
    edge_nodes_small_to_large = [edge_node_list[i] for i in edge_small_to_large_arg]
 
    num_edges = lenX - 1
 
    # Constructing a new MST from zero.
    MST_new = nx.Graph()
    for i in range(lenX):
        MST_new.add_node(i) 
        
    for i in range(num_edges):
        # Adding the edges one-by-one, from small to large, into the new MST.
        if i > 0:
            two_nodes_temp = edge_nodes_small_to_large[i - 1]
            MST_new.add_edge(two_nodes_temp[0], two_nodes_temp[1]) 

        edge_weight = edge_weight_small_to_large[i]
        edge_nodes = edge_nodes_small_to_large[i]

        tree1_nodes = list(nx.dfs_preorder_nodes(MST_new, source=edge_nodes[0]))
        tree2_nodes = list(nx.dfs_preorder_nodes(MST_new, source=edge_nodes[1]))

        idx1, idx2 = np.meshgrid(tree1_nodes, tree2_nodes, indexing="ij")
        mmj_matrix[idx1, idx2] = mmj_matrix[idx2, idx1] = edge_weight
    return mmj_matrix


In [5]:

data_id = 136

# Test the algorithm with 10 points:
X = test_data_145[data_id][:10]*100

print(f"Number of points in data X: {len(X)}" )
 
start = time.time()
X_mmj_matrix_algo_4_parallel_compu = cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(X)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used for computing MMJ matrix: {time_used}s" )

Number of points in data X: 10
Print the MST:
0 1 3.1
0 2 6.7
2 3 2.7
2 4 1.6
7 5 6.4
5 6 2.3
4 7 5.2
3 8 1.8
4 9 4.7
Time used for computing MMJ matrix: 0.953s


In [6]:
X_mmj_matrix_algo_4_parallel_compu

array([[0. , 3.1, 6.7, 6.7, 6.7, 6.7, 6.7, 6.7, 6.7, 6.7],
       [3.1, 0. , 6.7, 6.7, 6.7, 6.7, 6.7, 6.7, 6.7, 6.7],
       [6.7, 6.7, 0. , 2.7, 1.6, 6.4, 6.4, 5.2, 2.7, 4.7],
       [6.7, 6.7, 2.7, 0. , 2.7, 6.4, 6.4, 5.2, 1.8, 4.7],
       [6.7, 6.7, 1.6, 2.7, 0. , 6.4, 6.4, 5.2, 2.7, 4.7],
       [6.7, 6.7, 6.4, 6.4, 6.4, 0. , 2.3, 6.4, 6.4, 6.4],
       [6.7, 6.7, 6.4, 6.4, 6.4, 2.3, 0. , 6.4, 6.4, 6.4],
       [6.7, 6.7, 5.2, 5.2, 5.2, 6.4, 6.4, 0. , 5.2, 5.2],
       [6.7, 6.7, 2.7, 1.8, 2.7, 6.4, 6.4, 5.2, 0. , 4.7],
       [6.7, 6.7, 4.7, 4.7, 4.7, 6.4, 6.4, 5.2, 4.7, 0. ]])